In [1]:
!pip install ultralytics

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from ultralytics import YOLO
import json

with open("../outputs/incidents/stage1_frames.json") as f:
    stage1_frames = json.load(f)

model = YOLO("../models/best.pt")
print("Model loaded. Classes:", model.names)
print(f"Loaded {len(stage1_frames)} frames")

Model loaded. Classes: {0: 'D00_longitudinal_crack', 1: 'D10_transverse_crack', 2: 'D20_alligator_crack', 3: 'D40_pothole', 4: 'repair'}
Loaded 164 frames


In [3]:
def detect_anomalies(frames, model, conf_threshold=0.4):
    all_detections = []
    for frame in frames:
        results = model.predict(frame["frame_path"], conf=conf_threshold, verbose=False)[0]
        for box in results.boxes:
            cls_id = int(box.cls[0])
            cls_name = model.names[cls_id]
            confidence = float(box.conf[0])
            detection = {
                "frame": frame["frame_number"],
                "class": cls_name,
                "conf": round(confidence, 3),
                "bbox": box.xyxy[0].tolist(),
                "gps": frame["gps"],
                "weather": frame["weather"]
            }
            all_detections.append(detection)
    return all_detections

real_detections = detect_anomalies(stage1_frames, model)
print(f"Total real detections: {len(real_detections)}")
print(real_detections[:5])

Total real detections: 78
[{'frame': 0, 'class': 'D00_longitudinal_crack', 'conf': 0.476, 'bbox': [2596.204833984375, 949.4796752929688, 3203.80810546875, 1418.819580078125], 'gps': [13.0827, 80.2707], 'weather': {'condition': 'Clouds', 'description': 'overcast clouds', 'temp_c': 32.21, 'humidity': 72}}, {'frame': 0, 'class': 'D00_longitudinal_crack', 'conf': 0.456, 'bbox': [1570.4234619140625, 780.0858764648438, 1774.2132568359375, 1210.1617431640625], 'gps': [13.0827, 80.2707], 'weather': {'condition': 'Clouds', 'description': 'overcast clouds', 'temp_c': 32.21, 'humidity': 72}}, {'frame': 5, 'class': 'D00_longitudinal_crack', 'conf': 0.682, 'bbox': [2872.069580078125, 1120.431640625, 3308.40185546875, 1449.8939208984375], 'gps': [13.082745, 80.270726], 'weather': {'condition': 'Clouds', 'description': 'overcast clouds', 'temp_c': 32.21, 'humidity': 72}}, {'frame': 5, 'class': 'D00_longitudinal_crack', 'conf': 0.573, 'bbox': [1604.6480712890625, 727.121337890625, 1811.9344482421875, 